# Cordis raw EDA

In [14]:
import sys
sys.path.insert(0, '/home/lu72hip/DIGICHer/dh_pipeline/src')

import pandas as pd
from pathlib import Path
from lib.database.duck.create_connection import create_duck_connection
from utils.config.config_loader import get_query_config

# config = get_query_config()['cordis']['queries'][1]
# DB = Path(config['path_duck'])
DB = Path("/vast/lu72hip/data/duckdb/sources/cordis_raw.duckdb")

con = create_duck_connection(str(DB))
con.execute("SET memory_limit='16GB'")
con.execute("SET threads=8")
print(f'Connected to {DB}')

pd.set_option("display.max_rows", 50) # Show more rows  # or None for unlimited
pd.set_option("display.max_columns", None) # Show more columns  # None = show all
pd.set_option("display.max_colwidth", None) # Don't truncate column content  # None = full content
pd.set_option("display.width", None) # Wider display  # Auto-detect terminal width

Connected to /vast/lu72hip/data/duckdb/sources/cordis_raw.duckdb


## Stats

In [2]:
q = """ 
SELECT
    table_name,
    estimated_size as cnt
FROM duckdb_tables()
WHERE schema_name = 'main'
ORDER BY estimated_size DESC
"""
con.execute(q).df()

,table_name,cnt
0,j_researchoutput_person,1974269
1,person,1295289
2,j_project_topic,1054828
3,j_project_researchoutput,778828
4,researchoutput,778828
5,j_project_institution,741535
6,j_project_fundingprogramme,307456
7,weblink,265646
8,institution,177748
9,project,139925


In [ ]:
# Projects

q = """ 
SELECT * FROM institution
LIMIT 1;
"""

# Cordis.project columns:
# id	id_original	doi	title	acronym	status	start_date	end_date	ec_signature_date	total_cost	ec_max_contribution	objective	call_identifier	call_title	call_rcn	created_at	updated_at
# Cordis.j_project_institution columns:
# project_id	institution_id	institution_position	ec_contribution	net_ec_contribution	total_cost	type	organization_id	rcn
# Cordis.institution columns:
# 	id	legal_name	sme	url	short_name	vat_number	street	postbox	postalcode	city	country	geolocation	type_title	nuts_level_0	nuts_level_1	nuts_level_2	nuts_level_3	created_at	updated_at
con.execute(q).df()

,id,legal_name,sme,url,short_name,vat_number,street,postbox,postalcode,city,country,geolocation,type_title,nuts_level_0,nuts_level_1,nuts_level_2,nuts_level_3,created_at,updated_at
0,1,TEAGASC (AGRICULTURE AND FOOD DEVELOPMENT AUTHORITY),<NA>,None,None,None,Sandymount Ave. 19,None,4,DUBLIN,IE,"[-7.9034194, 53.0968609]",None,None,None,None,None,2026-03-13 11:55:46.283836,2026-03-13 11:55:46.283840


# Enrichment for OpenAIRE

* For project -> j_project_institution -> institution in cordis,
* i need to find the matching triplet in OpenAIRE,
* and copy cordis.j_project_institution.total_funding and is_coorinator to openaire.relation

In [ ]:
# Projects

q = """ 
SELECT * FROM project AS p
LEFT JOIN j_project_institution AS jpi ON jpi.project_id = p.id
LEFT JOIN institution AS i ON jpi.institution_id = i.id
LIMIT 1;
"""
con.execute(q).df()

In [ ]:
# junction project institution

q = """ 
SELECT * FROM j_project_institution
LIMIT 1;
"""
con.execute(q).df()

,project_id,institution_id,institution_position,ec_contribution,net_ec_contribution,total_cost,type,organization_id,rcn
0,1,1,1,NaN,NaN,NaN,participant,None,51846


# Triplet Matching Investigation

No direct OpenAire IDs in cordis duckdb. Matching plan:
1. **Projects**: `cordis.project.id_original` ↔ `core.project.grantId` (the EC grant code)
2. **Institutions**: `cordis.institution.legal_name` ↔ `core.organization.legalName` (exact after normalization, fuzzy fallback)

First we attach the core duckdb and probe both sides.

In [ ]:
# 1. Inspect project ID fields — what does id_original look like across eras?
q = """
SELECT id, id_original, title, start_date
FROM project
ORDER BY start_date DESC NULLS LAST
LIMIT 20
"""
con.execute(q).df()

In [ ]:
# 2. Attach core_v3 staging so we can probe both sides
CORE_DB = "/vast/lu72hip/data/duckdb/sources/openaire_staging.duckdb"
con.execute(f"ATTACH '{CORE_DB}' AS core (READ_ONLY)")

# Quick look at grantId in core to see what EC grant codes look like there
q = """
SELECT grantId, title, startDate
FROM core.project
WHERE grantId IS NOT NULL
ORDER BY startDate DESC NULLS LAST
LIMIT 20
"""
con.execute(q).df()

In [ ]:
# 3. How many cordis projects match core projects via id_original = grantId (exact)?
q = """
SELECT COUNT(*) AS matched_projects
FROM project AS cp
JOIN core.project AS op ON cp.id_original = op.grantId
"""
con.execute(q).df()

In [ ]:
# 4. For matched projects, how well do institution names match org names? (exact, case-insensitive)
q = """
WITH matched_projects AS (
    SELECT cp.id AS cordis_project_id, op.id AS core_project_id
    FROM project AS cp
    JOIN core.project AS op ON cp.id_original = op.grantId
),
cordis_triplets AS (
    SELECT
        mp.cordis_project_id,
        mp.core_project_id,
        i.id AS cordis_institution_id,
        i.legal_name,
        jpi.total_cost,
        jpi.type
    FROM matched_projects mp
    JOIN j_project_institution jpi ON jpi.project_id = mp.cordis_project_id
    JOIN institution i ON i.id = jpi.institution_id
)
SELECT
    COUNT(*) AS total_triplets,
    COUNT(DISTINCT core_project_id) AS distinct_projects,
    SUM(CASE WHEN EXISTS (
        SELECT 1 FROM core.organization co
        WHERE LOWER(TRIM(co.legalName)) = LOWER(TRIM(ct.legal_name))
    ) THEN 1 ELSE 0 END) AS exact_name_matches
FROM cordis_triplets ct
"""
con.execute(q).df()

In [ ]:
# 5. For the full triplet: project match + name match → how many relation rows can we enrich?
# This is the final join: core.relation WHERE source=core_project_id AND target=core_org_id
q = """
WITH matched_projects AS (
    SELECT cp.id AS cordis_project_id, op.id AS core_project_id
    FROM project AS cp
    JOIN core.project AS op ON cp.id_original = op.grantId
),
cordis_triplets AS (
    SELECT
        mp.core_project_id,
        co.id AS core_org_id,
        jpi.total_cost AS cordis_total_cost,
        jpi.type AS cordis_type
    FROM matched_projects mp
    JOIN j_project_institution jpi ON jpi.project_id = mp.cordis_project_id
    JOIN institution i ON i.id = jpi.institution_id
    JOIN core.organization co ON LOWER(TRIM(co.legalName)) = LOWER(TRIM(i.legal_name))
)
SELECT
    COUNT(*) AS enrichable_relation_rows
FROM core.relation r
JOIN cordis_triplets ct
    ON r.source = ct.core_project_id AND r.target = ct.core_org_id
"""
con.execute(q).df()

In [ ]:
# 6. Spot-check: show a few matched triplets side-by-side
q = """
WITH matched_projects AS (
    SELECT cp.id AS cordis_project_id, op.id AS core_project_id, cp.id_original AS grant_id
    FROM project AS cp
    JOIN core.project AS op ON cp.id_original = op.grantId
),
cordis_triplets AS (
    SELECT
        mp.grant_id,
        mp.core_project_id,
        i.legal_name AS cordis_legal_name,
        co.legalName AS core_legal_name,
        co.id AS core_org_id,
        jpi.total_cost AS cordis_total_cost,
        jpi.type AS cordis_type
    FROM matched_projects mp
    JOIN j_project_institution jpi ON jpi.project_id = mp.cordis_project_id
    JOIN institution i ON i.id = jpi.institution_id
    JOIN core.organization co ON LOWER(TRIM(co.legalName)) = LOWER(TRIM(i.legal_name))
)
SELECT * FROM cordis_triplets LIMIT 10
"""
con.execute(q).df()

In [ ]:
# 7. total_cost null rate in j_project_institution
q = """
SELECT
    COUNT(*) AS total,
    COUNT(total_cost) AS non_null_total_cost,
    COUNT(type) AS non_null_type,
    ROUND(COUNT(total_cost) * 100.0 / COUNT(*), 1) AS pct_total_cost,
    ROUND(COUNT(type) * 100.0 / COUNT(*), 1) AS pct_type
FROM j_project_institution
"""
con.execute(q).df()

In [ ]:
# 8. Relation direction check — is it always source=project, target=org? Or also reversed?
q = """
SELECT sourceType, targetType, COUNT(*) AS cnt
FROM core.relation
WHERE (sourceType = 'project' AND targetType = 'organization')
   OR (sourceType = 'organization' AND targetType = 'project')
GROUP BY 1, 2
ORDER BY cnt DESC
"""
con.execute(q).df()

In [13]:
# 9. Check for duplicate legalName in core.organization (would cause fanout in the join)
q = """
SELECT
    COUNT(*) AS total_orgs,
    COUNT(DISTINCT LOWER(TRIM(legalName))) AS distinct_names,
    total_orgs - distinct_names AS duplicate_name_count
FROM core.organization
WHERE legalName IS NOT NULL
"""
con.execute(q).df()

,total_orgs,distinct_names,duplicate_name_count
0,448161,389402,58759


In [14]:
# 10. Compare coverage of all money fields in j_project_institution
q = """
SELECT
    COUNT(*) AS total,
    COUNT(ec_contribution)     AS nn_ec_contribution,
    COUNT(net_ec_contribution) AS nn_net_ec_contribution,
    COUNT(total_cost)          AS nn_total_cost,
    ROUND(COUNT(ec_contribution)     * 100.0 / COUNT(*), 1) AS pct_ec_contribution,
    ROUND(COUNT(net_ec_contribution) * 100.0 / COUNT(*), 1) AS pct_net_ec_contribution,
    ROUND(COUNT(total_cost)          * 100.0 / COUNT(*), 1) AS pct_total_cost
FROM j_project_institution
"""
con.execute(q).df()

,total,nn_ec_contribution,nn_net_ec_contribution,nn_total_cost,pct_ec_contribution,pct_net_ec_contribution,pct_total_cost
0,741535,411767,300303,300122,55.5,40.5,40.5


In [15]:
# 11. Also check: are ec_contribution and total_cost always the same rows, or do they differ?
# And what does the data look like for a recent project with real values?
q = """
SELECT
    jpi.total_cost,
    jpi.ec_contribution,
    jpi.net_ec_contribution,
    jpi.type,
    i.legal_name,
    p.id_original AS grant_id
FROM j_project_institution jpi
JOIN institution i ON i.id = jpi.institution_id
JOIN project p ON p.id = jpi.project_id
WHERE jpi.ec_contribution IS NOT NULL
ORDER BY p.id DESC
LIMIT 10
"""
con.execute(q).df()

,total_cost,ec_contribution,net_ec_contribution,type,legal_name,grant_id
0,0.0,260347.921875,260347.921875,coordinator,"THE CHANCELLOR, MASTERS AND SCHOLARS OF THE UNIVERSITY OF OXFORD",101282279
1,0.0,202125.125000,202125.125000,coordinator,TECHNISCHE UNIVERSITAET MUENCHEN,101270681
2,0.0,209914.562500,209914.562500,coordinator,INSTITUTO DE ASTROFISICA DE CANARIAS,101264298
3,0.0,214344.718750,214344.718750,coordinator,UNIVERSITAT WIEN,101280885
4,0.0,263393.281250,263393.281250,coordinator,KOBENHAVNS UNIVERSITET,101282268
5,0.0,267930.906250,267930.906250,coordinator,UNIVERSITAT WIEN,101277446
6,0.0,267418.562500,267418.562500,coordinator,UNIVERSITETET I BERGEN,101279533
7,0.0,209483.281250,209483.281250,coordinator,UNIVERSITA DEGLI STUDI DI ROMA LA SAPIENZA,101211332
8,0.0,196976.156250,196976.156250,coordinator,VILNIAUS UNIVERSITETAS,101244517
9,0.0,191918.156250,191918.156250,coordinator,VYSOKA SKOLA CHEMICKO-TECHNOLOGICKA V PRAZE,101276940
